# trajectory-kit: `pos_writer` tutorial

`pos_writer` takes a structure file you've loaded into a `sim` (`.pdb` or
`.xyz`) plus a trajectory frame index, and writes out a **copy** of the
structure file with every atom's coordinates replaced by that frame's
coordinates. Everything else — atom names, residue ids, segment ids,
occupancy, B-factor, REMARK / CRYST1 / CONECT / END (PDB), the comment
line and any extra columns past z (XYZ velocities, forces) — is preserved
exactly.

By the end of this notebook you'll have:
1. Built a synthetic system (PDB + XYZ + DCD)
2. Stamped a single trajectory frame onto a PDB
3. Done the same for an XYZ
4. Used the path-wrapper for a one-shot script style
5. Used naming options (`output_dir`, `output_filepath`)
6. Dumped a sweep of frames in a loop


## 0. Setup — synthetic data files

We'll write a small PDB, an XYZ of the same atoms, and a 5-frame DCD into
a fresh temp directory. The DCD is built so each frame shifts every atom
by +1 Å along x — that makes the round-trip checks easy to read by eye.


In [ ]:
import struct
import tempfile
from pathlib import Path
import numpy as np

# Synthetic files are written to a fresh temp directory. The path is printed
# below so you can browse to it and inspect the raw files if you want to.
tmp = Path(tempfile.mkdtemp(prefix="trajkit_poswriter_"))

# 20-atom synthetic system: ALA-GLY-PRO + 2x TIP3 water.
# DCD: 5 frames, each frame shifts all atoms by +1 Angstrom along x.

pdb_text = """\
ATOM      1  N   ALA A   1       1.000   5.000   5.000  1.00  0.00      PROT
ATOM      2  HN  ALA A   1       0.100   5.800   5.300  1.00  0.00      PROT
ATOM      3  CA  ALA A   1       2.400   5.000   5.000  1.00  0.00      PROT
ATOM      4  CB  ALA A   1       3.000   6.300   5.200  1.00  0.00      PROT
ATOM      5  C   ALA A   1       3.200   3.900   4.600  1.00  0.00      PROT
ATOM      6  O   ALA A   1       2.800   2.800   4.400  1.00  0.00      PROT
ATOM      7  N   GLY A   2       4.600   4.100   4.500  1.00  0.00      PROT
ATOM      8  HN  GLY A   2       5.000   5.000   4.700  1.00  0.00      PROT
ATOM      9  CA  GLY A   2       5.800   3.200   4.100  1.00  0.00      PROT
ATOM     10  C   GLY A   2       7.200   3.500   4.000  1.00  0.00      PROT
ATOM     11  O   GLY A   2       7.700   4.600   4.200  1.00  0.00      PROT
ATOM     12  N   PRO A   3       8.100   2.500   3.700  1.00  0.00      PROT
ATOM     13  CA  PRO A   3       9.500   2.800   3.600  1.00  0.00      PROT
ATOM     14  C   PRO A   3      10.300   1.900   3.200  1.00  0.00      PROT
ATOM     15  OH2 TIP B   4      12.000   5.000   5.000  1.00  0.00      SOLV
ATOM     16  H1  TIP B   4      12.900   5.500   5.100  1.00  0.00      SOLV
ATOM     17  H2  TIP B   4      11.300   5.700   4.800  1.00  0.00      SOLV
ATOM     18  OH2 TIP B   5      14.000   3.000   5.500  1.00  0.00      SOLV
ATOM     19  H1  TIP B   5      14.800   3.600   5.400  1.00  0.00      SOLV
ATOM     20  H2  TIP B   5      13.400   3.700   5.200  1.00  0.00      SOLV
END
"""

# XYZ — same 20 atoms, free-format whitespace
xyz_lines = ["20\n", "synthetic ALA-GLY-PRO + 2x TIP3 water\n"]
for line in pdb_text.strip().split("\n"):
    if not line.startswith("ATOM"):
        continue
    name = line[12:16].strip()
    x = float(line[30:38]); y = float(line[38:46]); z = float(line[46:54])
    xyz_lines.append(f"{name:<6s} {x:10.4f} {y:10.4f} {z:10.4f}\n")
xyz_text = "".join(xyz_lines)

def write_dcd(path, n_atoms=20, n_frames=5):
    """5-frame DCD. Each frame shifts all atoms by +1 A along x."""
    def record(payload):
        n = struct.pack('<i', len(payload))
        return n + payload + n
    icntrl = [0] * 20; icntrl[0] = n_frames
    hdr = b'CORD' + struct.pack('<20i', *icntrl)
    hdr += b'\x00' * (84 - len(hdr))
    title = struct.pack('<i', 1) + b'synthetic 20-atom trajectory    '
    natom = struct.pack('<i', n_atoms)
    x0 = np.array([ 1.0, 0.1, 2.4, 3.0, 3.2, 2.8,
                    4.6, 5.0, 5.8, 7.2, 7.7, 8.1,
                    9.5,10.3,12.0,12.9,11.3,14.0,14.8,13.4], dtype=np.float32)
    y0 = np.array([ 5.0, 5.8, 5.0, 6.3, 3.9, 2.8,
                    4.1, 5.0, 3.2, 3.5, 4.6, 2.5,
                    2.8, 1.9, 5.0, 5.5, 5.7, 3.0, 3.6, 3.7], dtype=np.float32)
    z0 = np.array([ 5.0, 5.3, 5.0, 5.2, 4.6, 4.4,
                    4.5, 4.7, 4.1, 4.0, 4.2, 3.7,
                    3.6, 3.2, 5.0, 5.1, 4.8, 5.5, 5.4, 5.2], dtype=np.float32)
    with open(path, 'wb') as f:
        f.write(record(hdr))
        f.write(record(title))
        f.write(record(natom))
        for fi in range(n_frames):
            xs = x0 + float(fi)
            f.write(record(xs.tobytes()))
            f.write(record(y0.tobytes()))
            f.write(record(z0.tobytes()))

pdb_path = tmp / 'synth.pdb'
xyz_path = tmp / 'synth.xyz'
dcd_path = tmp / 'run.dcd'

pdb_path.write_text(pdb_text, encoding='ascii')
xyz_path.write_text(xyz_text, encoding='ascii')
write_dcd(dcd_path)

print(f"Synthetic files in: {tmp}")
print(f"  PDB:  {pdb_path.name}  (20 atoms)")
print(f"  XYZ:  {xyz_path.name}  (20 atoms)")
print(f"  DCD:  {dcd_path.name}  (5 frames; x shifts +1 A per frame)")


Synthetic files in: /tmp/trajkit_poswriter_4g4k6642
  PDB:  synth.pdb  (20 atoms)
  XYZ:  synth.xyz  (20 atoms)
  DCD:  run.dcd  (5 frames; x shifts +1 A per frame)


## 1. Stamping a trajectory frame onto the PDB

Load the PDB as typing and the DCD as trajectory, then call
`write_with_frame`. The output sits next to the source PDB by default,
named `<pdb_stem>_from_<traj_stem>_f{frame:06d>.pdb`.


In [2]:
from trajectory_kit import sim, pos_writer

s_pdb = sim(typing=pdb_path, trajectory=dcd_path)

out_pdb = pos_writer.write_with_frame(s_pdb, frame=3)
print("Wrote:", out_pdb.name)
print("  ->", out_pdb)


Wrote: synth_from_run_f000003.pdb
  -> /tmp/trajkit_poswriter_4g4k6642/synth_from_run_f000003.pdb


Verify the output: frame 3 should have x shifted by +3 Å for every
atom; y and z should be unchanged. We'll just print a few lines and check
the first atom by hand.

In [3]:
print("Source PDB — first 3 ATOM lines:")
for line in pdb_path.read_text().splitlines()[:3]:
    print(" ", line)

print()
print("Output PDB (frame 3) — first 3 ATOM lines:")
for line in out_pdb.read_text().splitlines()[:3]:
    print(" ", line)


Source PDB — first 3 ATOM lines:
  ATOM      1  N   ALA A   1       1.000   5.000   5.000  1.00  0.00      PROT
  ATOM      2  HN  ALA A   1       0.100   5.800   5.300  1.00  0.00      PROT
  ATOM      3  CA  ALA A   1       2.400   5.000   5.000  1.00  0.00      PROT

Output PDB (frame 3) — first 3 ATOM lines:
  ATOM      1  N   ALA A   1       4.000   5.000   5.000  1.00  0.00      PROT
  ATOM      2  HN  ALA A   1       3.100   5.800   5.300  1.00  0.00      PROT
  ATOM      3  CA  ALA A   1       5.400   5.000   5.000  1.00  0.00      PROT


Notice that the columns past `z` (occupancy, B-factor, segment id
`PROT`) and the columns before `x` (serial, atom name, residue, chain, resi)
are bit-identical. Only the x/y/z columns change.

## 2. Stamping a trajectory frame onto the XYZ

The XYZ writer follows the same pattern. The atom-count line and the
comment line pass through untouched. The element token, original
whitespace, and any extra trailing columns are preserved.

In [4]:
s_xyz = sim(typing=xyz_path, trajectory=dcd_path)

out_xyz = pos_writer.write_with_frame(s_xyz, frame=3)
print("Wrote:", out_xyz.name)

print()
print("Source XYZ — header + first 3 atom lines:")
for line in xyz_path.read_text().splitlines()[:5]:
    print(" ", line)

print()
print("Output XYZ (frame 3) — header + first 3 atom lines:")
for line in out_xyz.read_text().splitlines()[:5]:
    print(" ", line)


Wrote: synth_from_run_f000003.xyz

Source XYZ — header + first 3 atom lines:
  20
  synthetic ALA-GLY-PRO + 2x TIP3 water
  N          1.0000     5.0000     5.0000
  HN         0.1000     5.8000     5.3000
  CA         2.4000     5.0000     5.0000

Output XYZ (frame 3) — header + first 3 atom lines:
  20
  synthetic ALA-GLY-PRO + 2x TIP3 water
  N          4.0000     5.0000     5.0000
  HN         3.1000     5.8000     5.3000
  CA         5.4000     5.0000     5.0000


## 3. One-shot scripts — the path wrapper

If you don't already have a `sim` lying around, `write_with_frame_from_paths`
builds one for you internally and forwards.

In [5]:
quick_out = pos_writer.write_with_frame_from_paths(
    type_filepath       = pdb_path,
    trajectory_filepath = dcd_path,
    frame               = 0,
    output_dir          = tmp / "frames_quick",
)
print("Wrote:", quick_out)


Wrote: /tmp/trajkit_poswriter_4g4k6642/frames_quick/synth_from_run_f000000.pdb


## 4. Output naming

Three modes, in order of explicitness:

- **Default** — auto-named, written next to the source typing file.
- **`output_dir`** — auto-named, written to a directory you choose.
- **`output_filepath`** — exact path you specify. Extension must match the
  source typing file's extension; the writer preserves format, it doesn't
  convert.


In [6]:
# Default — next to the source PDB
o1 = pos_writer.write_with_frame(s_pdb, frame=1)
print("default:        ", o1.name, "  in", o1.parent.name)

# output_dir — auto-name in a directory of your choice
target_dir = tmp / "snapshots"
o2 = pos_writer.write_with_frame(s_pdb, frame=2, output_dir=target_dir)
print("output_dir:     ", o2.name, "  in", o2.parent.name)

# output_filepath — exact path
target_file = tmp / "my_snapshot.pdb"
o3 = pos_writer.write_with_frame(s_pdb, frame=2, output_filepath=target_file)
print("output_filepath:", o3.name, "  in", o3.parent.name)


default:         synth_from_run_f000001.pdb   in trajkit_poswriter_4g4k6642
output_dir:      synth_from_run_f000002.pdb   in snapshots
output_filepath: my_snapshot.pdb   in trajkit_poswriter_4g4k6642


## 5. Sweeping multiple frames

There's no built-in multi-frame mode — one call writes one file. To dump
a range, just loop. The 6-digit zero-padded frame index in the default
name means files sort correctly in any file browser.

In [7]:
sweep_dir = tmp / "sweep"

for f in range(5):
    pos_writer.write_with_frame(s_pdb, frame=f, output_dir=sweep_dir)

print("Files in sweep/:")
for p in sorted(sweep_dir.iterdir()):
    print(" ", p.name)


Files in sweep/:
  synth_from_run_f000000.pdb
  synth_from_run_f000001.pdb
  synth_from_run_f000002.pdb
  synth_from_run_f000003.pdb
  synth_from_run_f000004.pdb


## 6. Common errors

A quick tour of the guard rails. None of these will actually fire in this
notebook — they're shown here so you recognise the messages when they
do.

- **`pos_writer supports typing formats ['.pdb', '.xyz']`** — you loaded
  a `.mae` (or another format) as typing. Only PDB and XYZ are supported,
  because they're the formats `pos_writer` knows how to rewrite while
  preserving every other field.
- **`requires the sim to have a trajectory file loaded`** — without a
  trajectory there's nothing to swap in.
- **`Output path already exists`** — `pos_writer` refuses to overwrite.
  Delete the existing file or pick a new `output_dir` / `output_filepath`.
- **`output_filepath extension '.xyz' does not match the source typing
  file extension '.pdb'`** — the writer preserves the input format.


## 7. Verifying the output round-trips through `sim`

The cleanest sanity check: load the freshly-written file as a new `sim`
and confirm its coords match what `sim.positions(...)` would produce on
the original sim at that frame.

In [8]:
import numpy as np

# Original: positions for every atom at frame 3
original = s_pdb.positions(TRAJ_Q={'frame_interval': (3, 3)})

# Reload our frame-3 PDB and ask for its (static) positions
s_reload = sim(typing=out_pdb)
reloaded = s_reload.positions()

print("original.shape:", original.shape)
print("reloaded.shape:", reloaded.shape)
print("max abs diff:  ", float(np.max(np.abs(reloaded - original))))
print()
print("Match within float32 PDB precision (1e-3)?",
      np.allclose(reloaded, original, atol=1e-3))


original.shape: (1, 20, 3)
reloaded.shape: (1, 20, 3)
max abs diff:   0.0

Match within float32 PDB precision (1e-3)? True


That's it. `pos_writer` is intentionally narrow — one job, two
formats — so there's not much more API to learn. Anything beyond stamping
a single frame onto a structure scaffold is left to your own scripts on
top of the `sim` object.